# 02 · Two-stage anchor-based detector: Faster R-CNN + FPN

**Model.** torchvision `fasterrcnn_resnet50_fpn`, starting from COCO-pretrained weights; the box predictor is replaced for 8 classes + background.
- *Stage 1, Region Proposal Network (RPN):* scores anchors at every FPN location as object or background and regresses them into ~1000 class-agnostic proposals.
- *Stage 2, RoI head:* RoIAlign crops a 7×7 feature map for each proposal from the matching FPN level; two FC layers classify it into 8 classes + background and refine its box.

**Fixed protocol** (identical for every experiment; set in `PROTOCOL` below):

| | |
|---|---|
| Train data | `data_processed/train`, frame step from 01_preprocess, ignored regions grey-filled |
| Validation | 7 `val/` clips, same frame step, **original** images, `val/gt_eval_coco.json` |
| Metric | COCO mAP@[.5:.95], AP50, AP75, AP_small/medium/large, ≤ 500 detections per image (`common/evaluate.py`) |
| Input size | short side 720, long side ≤ 1280 (the model's internal resize) |
| Schedule | SGD, momentum 0.9, linear warm-up then cosine decay, AMP, fixed seed |
| Augmentation | `common/data.py::build_train_aug` (flip, mild scale, colour) |

**Experiments** (each changes one thing relative to the baseline):

| Name | Change | Why |
|---|---|---|
| `baseline` | torchvision defaults (anchors 32–512 px) | reference |
| `small_anchors` | anchors 16–256 px | EDA: many objects are smaller than the smallest default anchor |
| `more_rois` | 1024 RoIs per image, 50% positives, 2000 test proposals | dense frames: the default 128 positive RoIs per image can be fewer than the objects in it |
| `giou_loss` | GIoU box loss in the RoI head instead of Smooth L1 | IoU-based loss is scale-invariant, which suits a dataset where object size varies a lot |
| `tuned` | combination of the changes that helped | final tuned two-stage model |

**Outputs** go to `runs/faster_rcnn/<experiment>/`: `history.json` (per-epoch losses and metrics), `best.pt`, `val_dets.json`, `summary.json`.

## Setup

In [ ]:
import json
import math
import random
import sys
import time
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, fasterrcnn_resnet50_fpn
from torchvision.models.detection import roi_heads as tv_roi_heads
from torchvision.models.detection._utils import BoxCoder
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_iou, generalized_box_iou_loss
from tqdm.auto import tqdm

# --- Paths: local machine vs Kaggle ---
ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    SRC_DIR = Path("/kaggle/working/ai-hw2-multiobject-detection/src")      # git clone of our repo
    DATA_ROOT = Path("/kaggle/input/COMPETITION_FOLDER")                    # TODO: competition data
    PROC_ROOT = Path("/kaggle/input/PROCESSED_DATASET/data_processed")      # TODO: output of 01_preprocess
    RUNS_DIR = Path("/kaggle/working/runs/faster_rcnn")
else:
    SRC_DIR = Path.cwd()                                                    # this notebook lives in src/
    DATA_ROOT = Path("../multiple-object-video-detection-hw-2-ucu-ai-2026")
    PROC_ROOT = Path("../data_processed")
    RUNS_DIR = Path("../runs/faster_rcnn")

sys.path.insert(0, str(SRC_DIR))
from common.data import CLASSES, CocoDetection, ImageList, build_train_aug, collate, seed_worker_aug, to_model_input
from common.evaluate import evaluate_detections, measure_fps, to_coco_detections
from common.submission import write_submission

GT_FILE = PROC_ROOT / "val" / "gt_eval_coco.json"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_WANDB = False   # set True to log runs to Weights & Biases (needed for the visualisation bonus)
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

## Protocol and experiments

`PROTOCOL` is shared by every experiment and must not change between them. `BASE` holds the model settings of the baseline;
each experiment overrides only what it tests.

In [ ]:
PROTOCOL = dict(
    min_size=720, max_size=1280,     # model-internal resize: short side 720, long side <= 1280
    epochs=8,
    batch_size=4,
    lr=0.005,                        # linear scaling of torchvision's 0.02 for 16 images
    momentum=0.9, weight_decay=1e-4,
    warmup_iters=300,
    grad_clip=10.0,
    train_step=1,                    # extra subsampling of processed train frames (2 = every 2nd) for quick runs
    val_max_images=None,             # None = all val frames; an int = first N (quick checks only)
    num_workers=2,                   # each worker prefetches batches into RAM; keep low on 16 GB machines
    seed=42,
    amp=True,
    pretrained=True,
)

BASE = dict(
    anchor_sizes=((32,), (64,), (128,), (256,), (512,)),   # one size per FPN level P2..P6 (torchvision default)
    aspect_ratios=(0.5, 1.0, 2.0),                         # NOTE: torchvision defines these as height / width
    rpn_pre_nms_top_n_train=2000, rpn_post_nms_top_n_train=2000,
    rpn_pre_nms_top_n_test=1000, rpn_post_nms_top_n_test=1000,
    box_batch_size_per_image=512, box_positive_fraction=0.25,
    box_loss="smooth_l1",
    box_detections_per_img=500,      # submission cap
    box_score_thresh=0.001,          # keep low-score boxes: mAP only ranks them
    box_nms_thresh=0.5,
)

EXPERIMENTS = {
    "baseline": {},
    "small_anchors": {"anchor_sizes": ((16,), (32,), (64,), (128,), (256,))},
    "more_rois": {"box_batch_size_per_image": 1024, "box_positive_fraction": 0.5,
                  "rpn_pre_nms_top_n_test": 3000, "rpn_post_nms_top_n_test": 2000},
    "giou_loss": {"box_loss": "giou"},
    # Fill in after comparing the runs above: combine the changes that improved mAP.
    "tuned": {},
}

## Model

The custom `AnchorGenerator` keeps 3 aspect ratios per location, so the pretrained RPN head still loads; only the anchor sizes change.

In [ ]:
def build_model(cfg):
    weights = FasterRCNN_ResNet50_FPN_Weights.COCO_V1 if PROTOCOL["pretrained"] else None
    anchors = AnchorGenerator(sizes=cfg["anchor_sizes"],
                              aspect_ratios=(tuple(cfg["aspect_ratios"]),) * len(cfg["anchor_sizes"]))
    model = fasterrcnn_resnet50_fpn(
        weights=weights,
        weights_backbone=None,           # the COCO checkpoint already includes the backbone
        min_size=PROTOCOL["min_size"], max_size=PROTOCOL["max_size"],
        rpn_anchor_generator=anchors,
        rpn_pre_nms_top_n_train=cfg["rpn_pre_nms_top_n_train"], rpn_post_nms_top_n_train=cfg["rpn_post_nms_top_n_train"],
        rpn_pre_nms_top_n_test=cfg["rpn_pre_nms_top_n_test"], rpn_post_nms_top_n_test=cfg["rpn_post_nms_top_n_test"],
        box_batch_size_per_image=cfg["box_batch_size_per_image"], box_positive_fraction=cfg["box_positive_fraction"],
        box_detections_per_img=cfg["box_detections_per_img"], box_score_thresh=cfg["box_score_thresh"],
        box_nms_thresh=cfg["box_nms_thresh"],
    )
    # New classification/regression layer: 8 classes + background
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, len(CLASSES) + 1)
    return model

### GIoU box loss (loss-modification experiment)

torchvision's RoI head computes Smooth L1 on the encoded box deltas. To use GIoU we decode the predicted and target deltas into boxes and apply
`generalized_box_iou_loss`. Both deltas are relative to the same proposal, and IoU of axis-aligned boxes does not change when each axis is scaled
or shifted, so decoding against a unit reference box gives exactly the GIoU of the real boxes. The loss weight of 10 follows MMDetection's
Faster R-CNN GIoU config. The swap is done by replacing `torchvision.models.detection.roi_heads.fastrcnn_loss`.

In [ ]:
_SMOOTH_L1_LOSS = tv_roi_heads.fastrcnn_loss
_BOX_CODER = BoxCoder(weights=(10.0, 10.0, 5.0, 5.0))   # RoIHeads' default regression weights
GIOU_WEIGHT = 10.0

def giou_fastrcnn_loss(class_logits, box_regression, labels, regression_targets):
    labels = torch.cat(labels, dim=0)
    regression_targets = torch.cat(regression_targets, dim=0)
    cls_loss = F.cross_entropy(class_logits, labels)

    pos = torch.where(labels > 0)[0]
    n = class_logits.shape[0]
    box_regression = box_regression.reshape(n, box_regression.size(-1) // 4, 4)
    pred = box_regression[pos, labels[pos]].float()
    target = regression_targets[pos].float()
    ref = pred.new_tensor([[0.0, 0.0, 1.0, 1.0]]).expand(len(pos), 4)
    pred_boxes = _BOX_CODER.decode_single(pred, ref)
    target_boxes = _BOX_CODER.decode_single(target, ref)
    box_loss = GIOU_WEIGHT * generalized_box_iou_loss(pred_boxes, target_boxes, reduction="sum") / labels.numel()
    return cls_loss, box_loss

def set_box_loss(name):
    tv_roi_heads.fastrcnn_loss = giou_fastrcnn_loss if name == "giou" else _SMOOTH_L1_LOSS

## Data

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_loaders(seed):
    aug = build_train_aug(seed)
    train_ds = CocoDetection(PROC_ROOT / "train" / "annotations_coco.json", PROC_ROOT / "train", aug)
    if PROTOCOL["train_step"] > 1:
        train_ds = Subset(train_ds, range(0, len(train_ds), PROTOCOL["train_step"]))

    val_ds = CocoDetection(GT_FILE, DATA_ROOT)   # original val images, file_name is relative to DATA_ROOT
    if PROTOCOL["val_max_images"]:
        val_ds = Subset(val_ds, range(min(PROTOCOL["val_max_images"], len(val_ds))))
    base = val_ds.dataset if isinstance(val_ds, Subset) else val_ds
    idx = val_ds.indices if isinstance(val_ds, Subset) else range(len(base))
    val_ids = [base.images[i]["id"] for i in idx]

    nw = PROTOCOL["num_workers"]
    train_dl = DataLoader(train_ds, batch_size=PROTOCOL["batch_size"], shuffle=True, num_workers=nw,
                          collate_fn=collate, generator=torch.Generator().manual_seed(seed),
                          worker_init_fn=seed_worker_aug(aug, seed))   # no persistent workers: they would hold RAM during validation
    val_dl = DataLoader(val_ds, batch_size=2, shuffle=False, num_workers=nw, collate_fn=collate)
    return train_dl, val_dl, val_ids

## Training and evaluation loop

In [ ]:
def train_one_epoch(model, optimizer, scheduler, loader, scaler, epoch):
    model.train()
    sums, n = defaultdict(float), 0
    use_amp = PROTOCOL["amp"] and DEVICE.type == "cuda"
    for images, targets in tqdm(loader, desc=f"epoch {epoch}", leave=False):
        images = to_model_input(images, DEVICE)
        targets = [{k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in t.items()} for t in targets]
        with torch.autocast(DEVICE.type, enabled=use_amp):
            losses = model(images, targets)
            loss = sum(losses.values())
        if not torch.isfinite(loss):
            print("non-finite loss, batch skipped:", {k: v.item() for k, v in losses.items()})
            continue
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), PROTOCOL["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()   # a 'scheduler before optimizer' warning on the first step is expected with AMP
        for k, v in losses.items():
            sums[k] += v.item()
        sums["loss"] += loss.item()
        n += 1
    return {k: v / max(n, 1) for k, v in sums.items()}

@torch.no_grad()
def predict(model, loader):
    """COCO-format detections in original-image pixels."""
    model.eval()
    dets = []
    for images, targets in tqdm(loader, desc="predict", leave=False):
        outputs = model(to_model_input(images, DEVICE))
        dets += to_coco_detections(outputs, [t["image_id"] for t in targets])
    return dets

def warmup_cosine(total_iters, warmup_iters):
    def f(it):
        if it < warmup_iters:
            return 0.001 + (1 - 0.001) * it / warmup_iters
        progress = (it - warmup_iters) / max(1, total_iters - warmup_iters)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return f

In [ ]:
def run_experiment(name):
    cfg = {**BASE, **EXPERIMENTS[name]}
    out = RUNS_DIR / name
    out.mkdir(parents=True, exist_ok=True)
    seed_everything(PROTOCOL["seed"])
    set_box_loss(cfg["box_loss"])

    model = build_model(cfg).to(DEVICE)
    train_dl, val_dl, val_ids = make_loaders(PROTOCOL["seed"])
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=PROTOCOL["lr"], momentum=PROTOCOL["momentum"],
                                weight_decay=PROTOCOL["weight_decay"])
    total = PROTOCOL["epochs"] * len(train_dl)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_cosine(total, PROTOCOL["warmup_iters"]))
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=PROTOCOL["amp"] and DEVICE.type == "cuda")

    run = None
    if USE_WANDB:
        import wandb
        run = wandb.init(project="ucu-visdrone-detection", name=f"faster_rcnn/{name}",
                         config={**cfg, **PROTOCOL}, reinit=True)

    history, best = [], None
    print(f"[{name}] {len(train_dl.dataset)} train images, {len(val_ids)} val images, {total} iterations")
    for epoch in range(1, PROTOCOL["epochs"] + 1):
        t0 = time.time()
        losses = train_one_epoch(model, optimizer, scheduler, train_dl, scaler, epoch)
        dets = predict(model, val_dl)
        metrics = evaluate_detections(GT_FILE, dets, image_ids=val_ids)
        row = {"epoch": epoch, "minutes": (time.time() - t0) / 60, **losses, **metrics}
        history.append(row)
        print(f"  epoch {epoch}: loss {losses.get('loss', float('nan')):.3f} | mAP {metrics['mAP']:.4f} "
              f"AP50 {metrics['AP50']:.4f} AP_small {metrics['AP_small']:.4f} | {row['minutes']:.1f} min")
        if best is None or metrics["mAP"] > best["mAP"]:
            best = row
            torch.save(model.state_dict(), out / "best.pt")
            with open(out / "val_dets.json", "w") as f:
                json.dump(dets, f)
        with open(out / "history.json", "w") as f:
            json.dump(history, f, indent=1)
        if run:
            run.log(row)

    summary = {"name": name, "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in cfg.items()},
               "protocol": PROTOCOL, "best": best}
    with open(out / "summary.json", "w") as f:
        json.dump(summary, f, indent=1)
    if run:
        run.finish()
    return history

## Run experiments

Each run is a full training under the protocol. Run them one at a time (or list several) and they are saved to disk, so a restarted kernel
doesn't lose finished runs. Rough cost: on a T4, one epoch over ~4.8k frames at 1280 px takes around 10–15 min.

In [ ]:
RUN = ["baseline"]          # e.g. ["baseline", "small_anchors", "more_rois", "giou_loss"], later ["tuned"]
for name in RUN:
    run_experiment(name)

## Compare experiments

All experiments use the same data, protocol and evaluator, so their numbers are directly comparable.

In [ ]:
def load_runs():
    runs = {}
    for name in EXPERIMENTS:
        f = RUNS_DIR / name / "summary.json"
        if f.exists():
            runs[name] = json.loads(f.read_text())
    return runs

runs = load_runs()
cols = ["epoch", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"]
table = pd.DataFrame({n: {c: r["best"][c] for c in cols} for n, r in runs.items()}).T
table.style.format({c: "{:.4f}" for c in cols if c != "epoch"}).highlight_max(axis=0, subset=cols[1:])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for name in runs:
    h = pd.DataFrame(json.loads((RUNS_DIR / name / "history.json").read_text()))
    axes[0].plot(h.epoch, h["loss"], marker="o", label=name)
    axes[1].plot(h.epoch, h["mAP"], marker="o", label=name)
    axes[2].plot(h.epoch, h["AP_small"], marker="o", label=name)
for ax, t in zip(axes, ["train loss", "val mAP@[.5:.95]", "val AP_small"]):
    ax.set(xlabel="epoch", ylabel=t, title=t)
    ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Per-class AP of every run (rare classes are noisy: few distinct objects in val)
per_class = pd.DataFrame({n: {c: r["best"][f"AP_{c}"] for c in CLASSES} for n, r in runs.items()})
per_class.plot.bar(figsize=(12, 4), ylabel="AP@[.5:.95]", title="Per-class AP")
plt.show()
per_class.round(4)

**Observations:** _fill in after the runs: which changes helped, by how much, and on which object sizes._

## Inference speed

Images per second at batch size 1 on the current device, including the model's internal resize to 1280 px, measured on 50 val frames.

In [ ]:
def load_trained(name):
    cfg = {**BASE, **EXPERIMENTS[name]}
    model = build_model(cfg)
    model.load_state_dict(torch.load(RUNS_DIR / name / "best.pt", map_location="cpu"))
    return model.to(DEVICE).eval()

val_ds = CocoDetection(GT_FILE, DATA_ROOT)
speed_images = [val_ds[i][0] for i in range(0, len(val_ds), max(1, len(val_ds) // 50))][:50]
fps = {name: measure_fps(load_trained(name), speed_images, DEVICE) for name in runs}
table["FPS"] = pd.Series(fps)
table.round(4)

## What each stage contributes

For the selected model we measure, per ground-truth object, whether it is covered (IoU ≥ 0.5) by:
- the top 100 / 300 / 1000 **RPN proposals** (class-agnostic, before the second stage);
- the **final detections** (after RoI classification, box refinement and NMS), class-agnostic and with the correct class.

We also compare how well the best-matching proposal and the best-matching final box fit each object (mean IoU).
The gap between proposal recall and final recall shows what the second stage filters or loses; the IoU gain shows its box refinement.

In [ ]:
ANALYSE = "baseline"     # change to the tuned experiment once it exists

def size_bucket(boxes):
    area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return np.where(area < 32**2, "small", np.where(area < 96**2, "medium", "large"))

@torch.no_grad()
def stage_coverage(model, ds, n_images=150, top_ns=(100, 300, 1000)):
    model.eval()
    rows = []
    step = max(1, len(ds) // n_images)
    for i in tqdm(range(0, len(ds), step), desc="stage analysis"):
        img, tgt = ds[i]
        gt, gt_cls = tgt["boxes"].to(DEVICE), tgt["labels"].to(DEVICE)
        x = to_model_input([img], DEVICE)
        if len(gt) == 0:
            continue
        # Stage 1: proposals, mapped back to original-image pixels
        images, _ = model.transform(x)
        feats = model.backbone(images.tensors)
        proposals, _ = model.rpn(images, feats)
        h, w = images.image_sizes[0]
        scale = torch.tensor([w / img.shape[2], h / img.shape[1]] * 2, device=DEVICE)
        props = proposals[0] / scale                       # sorted by objectness
        # Stage 2: final detections
        det = model(x)[0]

        rec = {"bucket": size_bucket(gt.cpu().numpy())}
        for n in top_ns:
            iou = box_iou(gt, props[:n]).max(1).values if len(props) else torch.zeros(len(gt), device=DEVICE)
            rec[f"proposals@{n}"] = (iou >= 0.5).cpu().numpy()
        rec["proposal_best_iou"] = box_iou(gt, props).max(1).values.cpu().numpy()
        if len(det["boxes"]):
            iou_det = box_iou(gt, det["boxes"])
            rec["final (any class)"] = (iou_det.max(1).values >= 0.5).cpu().numpy()
            same_cls = gt_cls[:, None] == det["labels"][None, :]
            rec["final (correct class)"] = ((iou_det * same_cls).max(1).values >= 0.5).cpu().numpy()
            rec["final_best_iou"] = iou_det.max(1).values.cpu().numpy()
        else:
            for k in ["final (any class)", "final (correct class)"]:
                rec[k] = np.zeros(len(gt), bool)
            rec["final_best_iou"] = np.zeros(len(gt))
        rows.append(pd.DataFrame(rec))
    return pd.concat(rows, ignore_index=True)

cov = stage_coverage(load_trained(ANALYSE), val_ds)
recall_cols = [c for c in cov.columns if c.startswith(("proposals@", "final ("))]
summary_tbl = pd.concat([cov.groupby("bucket")[recall_cols].mean().T, cov[recall_cols].mean().rename("all")], axis=1)
print("Recall at IoU >= 0.5 (rows: stage, columns: object size)")
display(summary_tbl[["small", "medium", "large", "all"]].round(3))
matched = cov[cov["final (any class)"]]
print(f"Mean best IoU for objects found by both stages: proposals {matched.proposal_best_iou.mean():.3f} "
      f"-> final boxes {matched.final_best_iou.mean():.3f}")

**Observations:** _fill in: how much recall the RPN provides, how much the second stage keeps, and how much it improves localisation, per size._

## Failure analysis

Detections with score ≥ `SCORE_THR` are matched greedily to ground truth (IoU ≥ 0.5). Each frame gets counts of:
- **missed** objects (false negatives),
- **false positives** that don't overlap any object (detections lying mostly inside ignored regions are not counted),
- **wrong class**: the box matches an object (IoU ≥ 0.5) of another class.

The three cells below show the frames with the most misses, the most false positives and the most class confusions.
Colours: green = correct, red = missed object, yellow = false positive, magenta = wrong class.

In [ ]:
SCORE_THR = 0.3
gt_coco = json.loads(GT_FILE.read_text())
gt_by_img, ign_by_img = defaultdict(list), defaultdict(list)
for a in gt_coco["annotations"]:
    (ign_by_img if a["iscrowd"] else gt_by_img)[a["image_id"]].append(a)
images_by_id = {im["id"]: im for im in gt_coco["images"]}

def xywh_to_xyxy(b):
    b = np.asarray(b, float).reshape(-1, 4)
    return np.c_[b[:, :2], b[:, :2] + b[:, 2:]]

def frame_errors(dets_by_img, img_id):
    gts = gt_by_img.get(img_id, [])
    gt_boxes = torch.tensor(xywh_to_xyxy([a["bbox"] for a in gts]))
    gt_cls = np.array([a["category_id"] for a in gts])
    ds = sorted([d for d in dets_by_img.get(img_id, []) if d["score"] >= SCORE_THR], key=lambda d: -d["score"])
    det_boxes = torch.tensor(xywh_to_xyxy([d["bbox"] for d in ds]))
    det_cls = np.array([d["category_id"] for d in ds])
    iou = box_iou(det_boxes, gt_boxes).numpy() if len(ds) and len(gts) else np.zeros((len(ds), len(gts)))

    # Ignored regions (one copy per class in the GT; deduplicate by box)
    ign = {tuple(a["bbox"]) for a in ign_by_img.get(img_id, [])}
    ign_boxes = torch.tensor(xywh_to_xyxy(list(ign))) if ign else torch.zeros((0, 4))

    matched_gt = np.zeros(len(gts), bool)
    status = []
    for i in range(len(ds)):
        same = np.where((det_cls[i] == gt_cls) & ~matched_gt, iou[i], 0) if len(gts) else np.array([])
        if len(same) and same.max() >= 0.5:
            matched_gt[same.argmax()] = True
            status.append("tp")
        elif len(gts) and iou[i].max() >= 0.5:
            status.append("wrong_class")
        else:
            inside = 0.0
            if len(ign_boxes):
                b = det_boxes[i:i + 1]
                lt = torch.max(b[:, None, :2], ign_boxes[None, :, :2]); rb = torch.min(b[:, None, 2:], ign_boxes[None, :, 2:])
                inter = (rb - lt).clamp(min=0).prod(-1)
                inside = float(inter.sum() / ((b[:, 2:] - b[:, :2]).prod(-1) + 1e-9))
            status.append("ignored" if inside >= 0.5 else "fp")
    return dict(gt_boxes=gt_boxes.numpy(), gt_cls=gt_cls, matched_gt=matched_gt,
                det_boxes=det_boxes.numpy(), det_cls=det_cls, status=status)

dets = json.loads((RUNS_DIR / ANALYSE / "val_dets.json").read_text())
dets_by_img = defaultdict(list)
for d in dets:
    dets_by_img[d["image_id"]].append(d)

rows = []
for img_id in images_by_id:
    e = frame_errors(dets_by_img, img_id)
    rows.append({"image_id": img_id, "video": images_by_id[img_id]["video"], "n_gt": len(e["gt_cls"]),
                 "missed": int((~e["matched_gt"]).sum()), "fp": e["status"].count("fp"),
                 "wrong_class": e["status"].count("wrong_class")})
errors = pd.DataFrame(rows)
print("Totals over val:", errors[["n_gt", "missed", "fp", "wrong_class"]].sum().to_dict())
errors.groupby("video")[["n_gt", "missed", "fp", "wrong_class"]].sum()

In [ ]:
def show_frame(img_id, title=""):
    e = frame_errors(dets_by_img, img_id)
    info = images_by_id[img_id]
    img = cv2.imread(str(DATA_ROOT / info["file_name"]))
    for box, ok in zip(e["gt_boxes"], e["matched_gt"]):
        if not ok:
            cv2.rectangle(img, tuple(map(int, box[:2])), tuple(map(int, box[2:])), (0, 0, 255), 2)
    colors = {"tp": (0, 200, 0), "fp": (0, 255, 255), "wrong_class": (255, 0, 255)}
    for box, c, s in zip(e["det_boxes"], e["det_cls"], e["status"]):
        if s in colors:
            cv2.rectangle(img, tuple(map(int, box[:2])), tuple(map(int, box[2:])), colors[s], 1)
            if s == "wrong_class":
                cv2.putText(img, CLASSES[c - 1], tuple(map(int, box[:2])), cv2.FONT_HERSHEY_SIMPLEX, 0.4, colors[s], 1)
    plt.figure(figsize=(16, 9))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis("off")
    plt.title(f"{title} | {info['video']} frame {info['frame_id']}")
    plt.show()

for col, title in [("missed", "Most missed objects"), ("fp", "Most false positives"), ("wrong_class", "Most class confusions")]:
    worst = errors.sort_values(col, ascending=False).iloc[0]
    show_frame(int(worst.image_id), f"{title}: {int(worst[col])} of {int(worst.n_gt)} objects")

In [ ]:
# Which objects are missed, and which classes get confused?
miss_rows, conf_rows = [], []
for img_id in images_by_id:
    e = frame_errors(dets_by_img, img_id)
    for box, c, ok in zip(e["gt_boxes"], e["gt_cls"], e["matched_gt"]):
        miss_rows.append({"cls": CLASSES[c - 1], "size": size_bucket(box[None])[0], "missed": not ok})
    for box, c, s in zip(e["det_boxes"], e["det_cls"], e["status"]):
        if s == "wrong_class":
            j = box_iou(torch.tensor(box[None]), torch.tensor(e["gt_boxes"])).argmax().item()
            conf_rows.append({"true": CLASSES[e["gt_cls"][j] - 1], "predicted": CLASSES[c - 1]})
miss = pd.DataFrame(miss_rows)
print("Miss rate by class and size:")
display(miss.pivot_table(index="cls", columns="size", values="missed", aggfunc="mean").reindex(CLASSES).round(3))
if conf_rows:
    print("Class confusions (true -> predicted):")
    display(pd.crosstab(pd.DataFrame(conf_rows)["true"], pd.DataFrame(conf_rows)["predicted"]))

**Failure cases:** _describe at least three, with plausible causes (e.g. tiny objects below the smallest anchor, dense crowds suppressed by NMS, van/car confusion, motion blur)._

## Test predictions and submission

Runs the selected model on every test frame (original resolution, same internal resize) and writes the competition CSV.

In [ ]:
SUBMIT = "baseline"      # experiment to submit

@torch.no_grad()
def predict_test(model):
    test = pd.read_csv(DATA_ROOT / "test_images.csv")
    ds = ImageList([DATA_ROOT / f for f in test["file_name"]], test["image_id"])
    dl = DataLoader(ds, batch_size=2, num_workers=PROTOCOL["num_workers"], collate_fn=collate)
    preds = {}
    for images, keys in tqdm(dl, desc="test"):
        for out, key in zip(model(to_model_input(images, DEVICE)), keys):
            b = out["boxes"].cpu().numpy()
            preds[key] = [(int(c) - 1, float(s), x1, y1, x2 - x1, y2 - y1)
                          for (x1, y1, x2, y2), s, c in zip(b, out["scores"].cpu().numpy(), out["labels"].cpu().numpy())]
    return preds

sub_path = RUNS_DIR / SUBMIT / "submission.csv"
sub = write_submission(predict_test(load_trained(SUBMIT)), DATA_ROOT / "test_images.csv", sub_path)
print(f"Wrote {sub_path}: {len(sub)} rows, {sub.PredictionString.str.len().gt(0).sum()} with detections")
sub.head()